# 14 — Model Evaluation & Conformal Uncertainty

**Objective:** Final evaluation on the **Locked 2015 Chronological Test Set**.
- Calibration of 90% split-conformal prediction intervals using 2014 validation errors.
- Evaluate marginal and simultaneous empirical coverage on 2015 test data.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.data.loader import FEATURES_A, TARGETS_A
from src.data.splitting import temporal_split
from src.evaluation.conformal import conformal_quantiles, interval_metrics
import joblib


## 1. Conformal Uncertainty Metrics on Locked 2015 Test Set


In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")
dev, cal, test = temporal_split(df)

model_path = ROOT / "artifacts" / "models" / "A_emissions" / "HistGradientBoosting.joblib"
if not model_path.exists():
    model_path = ROOT / "models" / "A_emissions" / "HistGradientBoosting.joblib"

bundle = joblib.load(model_path)
estimator = bundle["estimator"]

pred_cal = estimator.predict(cal[FEATURES_A])
pred_test = estimator.predict(test[FEATURES_A])

q = conformal_quantiles(cal[TARGETS_A].values, pred_cal, alpha=0.10)
ci = interval_metrics(test[TARGETS_A].values, pred_test, q)

uncertainty_records = [{
    "model": "HistGradientBoosting",
    "track": "A_emissions",
    "nominal_coverage": 0.90,
    "simultaneous_coverage": ci["simultaneous_coverage"],
    "target": TARGETS_A[j],
    "marginal_coverage": ci["marginal_coverage"][j],
    "interval_width": ci["mean_interval_width"][j]
} for j in range(len(TARGETS_A))]

df_unc = pd.DataFrame(uncertainty_records)
print(df_unc.to_string(index=False))

out_unc = ROOT / "experiments" / "uncertainty_metrics.csv"
df_unc.to_csv(out_unc, index=False)
print(f"Saved uncertainty metrics to {out_unc.relative_to(ROOT)}")
print("Conformal uncertainty evaluation COMPLETE.")


               model       track  nominal_coverage  simultaneous_coverage target  marginal_coverage  interval_width
HistGradientBoosting A_emissions               0.9               0.574485     CO           0.876219        3.316499
HistGradientBoosting A_emissions               0.9               0.574485    NOX           0.672535       30.401558
Saved uncertainty metrics to experiments\uncertainty_metrics.csv
Conformal uncertainty evaluation COMPLETE.
